Εξόρυξη δεδομένων: Άσκηση 1Β

ΟΜΑΔΑ: Αριστείδης Νικολακόπουλος (cs5308), Χρήστος  Γιαμαλής (ma12834)

USING 1 FREE PASS

Step 0: Imports , data loading and data cleaning


In [ ]:
#Εξόρυξη δεδομένων: Άσκηση 2

#ΟΜΑΔΑ: Αριστείδης Νικολακόπουλος (cs5308), Χρήστος  Γιαμαλής (ma12834)

#*******************
#USING 1 FREE PASS
#*******************

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, csc_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
import time
import matplotlib.pyplot as plt

try:
    train_df = pd.read_csv('data_train.csv')
    test_df = pd.read_csv('data_test.csv')
except FileNotFoundError:
    print("Error: Files 'data_train.csv' or 'data_test.csv' not found.")

train_df.columns = ['userId', 'movieId', 'rating']
test_df.columns = ['userId', 'movieId', 'rating']

n_users = max(train_df['userId'].max(), test_df['userId'].max()) + 1
n_items = max(train_df['movieId'].max(), test_df['movieId'].max()) + 1

print(f"Data Loaded. Users: {n_users}, Movies: {n_items}")

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

Data Loaded. Users: 671, Movies: 9066


Step 1: User Average (UA) & Item Average (IA)

In [9]:
print("\n--- Step 1: Baseline Algorithms ---")

user_means = train_df.groupby('userId')['rating'].mean()
global_mean = train_df['rating'].mean()

ua_preds = test_df['userId'].map(user_means).fillna(global_mean)
ua_rmse = calculate_rmse(test_df['rating'], ua_preds)
print(f"User Average (UA) RMSE: {ua_rmse:.4f}")

item_means = train_df.groupby('movieId')['rating'].mean()

ia_preds = test_df['movieId'].map(item_means).fillna(global_mean)
ia_rmse = calculate_rmse(test_df['rating'], ia_preds)
print(f"Item Average (IA) RMSE: {ia_rmse:.4f}")


--- Step 1: Baseline Algorithms ---
User Average (UA) RMSE: 0.9613
Item Average (IA) RMSE: 0.9860


Step 2: Singular Value Decomposition (SVD)

In [10]:
print("\n--- Step 2: SVD ---")


R = csr_matrix((train_df['rating'], (train_df['userId'], train_df['movieId'])),
               shape=(n_users, n_items), dtype=float)


K_SVD = 20
U, Sigma, Vt = svds(R, k=K_SVD)


U = np.flip(U, axis=1)
Sigma = np.flip(Sigma)
Vt = np.flip(Vt, axis=0)

svd_rmse_list = []
k_values_svd = range(1, K_SVD + 1)

test_users = test_df['userId'].values
test_items = test_df['movieId'].values
true_ratings = test_df['rating'].values

for k in k_values_svd:
    Uk = U[:, :k]
    Sk = np.diag(Sigma[:k])
    Vtk = Vt[:k, :]

    USk = Uk @ Sk
    user_vecs = USk[test_users, :]
    item_vecs = Vtk[:, test_items].T

    predictions = np.sum(user_vecs * item_vecs, axis=1)
    predictions = np.clip(predictions, 0, 5) # [cite: 53]

    rmse = calculate_rmse(true_ratings, predictions)
    svd_rmse_list.append(rmse)
    print(f"SVD k={k}: RMSE={rmse:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(k_values_svd, svd_rmse_list, marker='o')
plt.title('SVD Performance: RMSE vs k')
plt.xlabel('k (Rank)')
plt.ylabel('RMSE')
plt.grid(True)
plt.savefig('svd_rmse_plot.png')
plt.close()

best_svd_idx = np.argmin(svd_rmse_list)
min_svd_rmse = svd_rmse_list[best_svd_idx]
print(f"Best SVD RMSE: {min_svd_rmse:.4f} at k={k_values_svd[best_svd_idx]}")


--- Step 2: SVD ---
SVD k=1: RMSE=3.1693
SVD k=2: RMSE=3.0719
SVD k=3: RMSE=3.0101
SVD k=4: RMSE=2.9804
SVD k=5: RMSE=2.9613
SVD k=6: RMSE=2.9424
SVD k=7: RMSE=2.9221
SVD k=8: RMSE=2.9231
SVD k=9: RMSE=2.9179
SVD k=10: RMSE=2.9219
SVD k=11: RMSE=2.9156
SVD k=12: RMSE=2.9226
SVD k=13: RMSE=2.9159
SVD k=14: RMSE=2.9103
SVD k=15: RMSE=2.9087
SVD k=16: RMSE=2.9172
SVD k=17: RMSE=2.9228
SVD k=18: RMSE=2.9271
SVD k=19: RMSE=2.9327
SVD k=20: RMSE=2.9360
Best SVD RMSE: 2.9087 at k=15


Step 4


In [11]:
print("\n--- Step 4: UCF (Optimized) ---")


--- Step 4: UCF (Optimized) ---


4.1: Simple Functions

In [12]:
# Step 4.1

def user_mean(R_mat, u):
    # R_mat must be CSR for efficient row slicing
    data = R_mat[u, :].data
    return np.mean(data) if data.size > 0 else 0

def item_users(R_mat, m):
    # R_mat must be CSC for efficient column slicing
    return R_mat[:, m].indices

def similar_users(R_mat, u, candidates, k):
    if len(candidates) == 0:
        return np.array([]), np.array([])

    u_vec = R_mat[u, :]
    candidate_vecs = R_mat[candidates, :]

    sims = cosine_similarity(u_vec, candidate_vecs).flatten()

    sorted_indices = np.argsort(sims)[::-1]

    top_indices = sorted_indices[:k]
    return candidates[top_indices], sims[top_indices]

def compute_score(ratings, similarities):
    return np.dot(ratings, similarities) / np.sum(similarities)


4.2 Running UCF

In [13]:
# S4.3: Optimized UCF Function
def run_ucf(train_R, test_df, k_list):
    R_csr = train_R.tocsr()
    R_csc = train_R.tocsc()

    max_k = max(k_list)
    results = {k: [] for k in k_list}
    test_data = test_df.values

    start_time = time.time()

    for row in test_data:
        u, m = int(row[0]), int(row[1])

        # Find users who rated m
        users_who_rated = item_users(R_csc, m)
        # Exclude self
        users_who_rated = users_who_rated[users_who_rated != u]

        u_avg = user_mean(R_csr, u)

        if len(users_who_rated) == 0:
            for k in k_list:
                results[k].append(u_avg)
            continue

        # 2. Compute Similarities for max_k
        top_users, top_sims = similar_users(R_csr, u, users_who_rated, max_k)

        # Get ratings for these top users
        top_ratings = np.array([R_csr[user_id, m] for user_id in top_users])

        # 3. Compute Scores for all k
        for k in k_list:
            slice_k = min(len(top_users), k)

            if slice_k == 0:
                pred = u_avg
            else:
                k_sims = top_sims[:slice_k]
                k_ratings = top_ratings[:slice_k]

                if np.sum(k_sims) == 0:
                     pred = 0 #
                else:
                    pred = compute_score(k_ratings, k_sims) # [cite: 79]

            results[k].append(pred)

    duration = time.time() - start_time
    print(f"UCF Execution Time: {duration:.2f} seconds")

    rmse_results = {}
    true_vals = test_df['rating'].values

    for k, preds in results.items():
        preds = np.clip(preds, 0, 5)
        rmse_val = calculate_rmse(true_vals, preds)
        rmse_results[k] = rmse_val
        print(f"UCF k={k}: RMSE={rmse_val:.4f}")

    return rmse_results

k_ucf_list = [1, 2, 3, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

ucf_rmses = run_ucf(R, test_df, k_ucf_list)

plt.figure(figsize=(8, 4))
plt.plot(list(ucf_rmses.keys()), list(ucf_rmses.values()), marker='o', color='green')
plt.title('UCF Performance: RMSE vs k')
plt.xlabel('k (Neighbors)')
plt.ylabel('RMSE')
plt.grid(True)
plt.savefig('ucf_rmse_plot.png')
plt.close()

best_ucf_k = min(ucf_rmses, key=ucf_rmses.get)
best_ucf_rmse = ucf_rmses[best_ucf_k]
print(f"Best UCF RMSE: {best_ucf_rmse:.4f} at k={best_ucf_k}")

UCF Execution Time: 6.93 seconds
UCF k=1: RMSE=1.2267
UCF k=2: RMSE=1.0821
UCF k=3: RMSE=1.0366
UCF k=5: RMSE=0.9999
UCF k=10: RMSE=0.9770
UCF k=20: RMSE=0.9712
UCF k=30: RMSE=0.9700
UCF k=40: RMSE=0.9706
UCF k=50: RMSE=0.9708
UCF k=60: RMSE=0.9712
UCF k=70: RMSE=0.9711
UCF k=80: RMSE=0.9712
UCF k=90: RMSE=0.9715
UCF k=100: RMSE=0.9717
Best UCF RMSE: 0.9700 at k=30


Results

In [14]:
results_data = {
    'Algorithm': [
        'User Average',
        'Item Average',
        f'SVD (k={k_values_svd[best_svd_idx]})',
        f'UCF (k={best_ucf_k})'
    ],
    'Best RMSE': [
        ua_rmse,
        ia_rmse,
        min_svd_rmse,
        best_ucf_rmse
    ]
}

results_df = pd.DataFrame(results_data)
print(results_df.to_string(index=False))

   Algorithm  Best RMSE
User Average   0.961285
Item Average   0.986019
  SVD (k=15)   2.908652
  UCF (k=30)   0.970042
